## Milestone 3
### Jennifer Tsang, Nicole Link

In [1]:
import os
from pathlib import Path
import pandas as pd
import duckdb
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers import EnsembleRetriever
import pickle
import transformers
from transformers import pipeline
import torch
from langchain_groq import ChatGroq

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if Path.cwd().name != "DSCI_575_project_jt8919_nicolelink33":
    project_root = Path.cwd().parent 
    os.chdir(project_root)
    
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/Nicole/MDS/575/DSCI_575_project_jt8919_nicolelink33


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from src.utils import simple_tokenize, display_results
from src.bm25 import search_bm25_with_scores
from src.semantic import search_semantic_with_scores
from src.hybrid import get_hybrid_retriever
from src.rag_pipeline import build_context, get_prompt_template, get_rag_chain

In [5]:
DATA_DIR = Path("data")
CATEGORY = "Arts_Crafts_and_Sewing"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"
bm25_path = "models/bm25_metadata_index_big.pkl"
semantic_path = "models/faiss_index_big"
OUTPUT = 'data/processed/stratified_sample_big.parquet'

In [6]:
print(REVIEWS_URL)

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Arts_Crafts_and_Sewing.jsonl.gz


## 0. Loading and Merging the Datasets

### Data Loading

In [7]:
c2 = duckdb.connect()

In [8]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,Received damaged. Has hole in mold!,[[VIDEOID:2a7ad2a91afb1e17ff4a1c143f7e10a2]] R...,[{'small_image_url': 'https://m.media-amazon.c...,B095RKB9N3,B095RXT585,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1661111719157,0,True
1,3.0,3rd one arrived scratched/dented.,3rd one that arrived damaged! I give up!,[{'small_image_url': 'https://m.media-amazon.c...,B08PNNKNSQ,B07QXN7TMP,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1655614875170,0,True
2,1.0,Abominable. NOT COLOR SHIFT! False Advertising!,These are regular mica powders! One appears t...,[{'small_image_url': 'https://m.media-amazon.c...,B094DH11SH,B094DH11SH,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1654510930828,0,True
3,2.0,Used 2x in 6 weeks dry as a bone.,[[VIDEOID:8644a01103c90ab8f83af816b83881be]] L...,[{'small_image_url': 'https://m.media-amazon.c...,B089YSSFD6,B087D548ZY,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1648170476726,0,True
4,1.0,Replaced & still Defective stamps!,Lowered from a 3 star review to a 1 star bc th...,[{'small_image_url': 'https://m.media-amazon.c...,B08Z47RG56,B08Z47RG56,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1644619507972,1,True


In [9]:
# executes right over the internet -- i just want five rows to preview so doesn't take long
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Amazon Home,Trapp Home Fragrance Wax Melts - No. 13 Bob's ...,4.4,108,[Includes (2) No. 13 Bob's Flower Shoppe Trapp...,[The No. 13 Bob's Flower Shoppe Home Fragrance...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Trapp,"[Arts, Crafts & Sewing, Crafting, Candle Makin...","{'Product Dimensions': '""6.75 x 3.4 x 1 inches...",B013W3MPCW,None,None,<NA>
1,"Arts, Crafts & Sewing",JESEP YONG 4packs Plastic Organizer Box 24 Gri...,4.3,103,[MATERIAL & COLOR: These storage boxes are mad...,[SIZE & PACKING: the outer box size is 7.5'' L...,12.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'craft organizers and storage', 'ur...",JESEP YONG,"[Arts, Crafts & Sewing, Organization, Storage ...","{'Brand': '""JESEP YONG""', 'Color': '""Clear""', ...",B09B59XWTG,None,None,<NA>
2,Pet Supplies,"JIESHU 2-Pack Foam Holders for centerpieces, B...",3.6,15,[The 6 suction cups at the bottom of the foam ...,[Size: 9.5 * 4.5 * 2.7 inches Ideal for manual...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],JIESHU,"[Arts, Crafts & Sewing, Crafting, Floral Arran...","{'Product Dimensions': '""9.5 x 4.5 x 2.7 inche...",B0915N2NX1,None,None,<NA>
3,"Arts, Crafts & Sewing",Metal Swivel Lobster Claw Clasp Split Ring Key...,3.2,35,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],NaN,"[Arts, Crafts & Sewing, Beading & Jewelry Maki...","{'Product Dimensions': '""4.1 x 3.7 x 0.6 inche...",B008SNYJSU,None,None,<NA>
4,"Arts, Crafts & Sewing",Sizzix Birdcage Favor Thinlits Gift Box by Dav...,4.6,464,[Thinlits dies allow you to cut intricate desi...,[Thinlits create dazzling detailed shapes for ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sizzix,"[Arts, Crafts & Sewing, Scrapbooking & Stampin...","{'Product Dimensions': '""5.38 x 4.38 x 0.13 in...",B07Z9FH9PR,None,None,<NA>


#### Download and convert to parquet

In [10]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{REVIEWS_URL}')  LIMIT 20000)
      TO 'data/raw/reviews_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [11]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO 'data/raw/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

#### Merge the two files

In [12]:
c2.execute("""
    COPY (
        SELECT r.rating, r.title, r.text, r.parent_asin, r.verified_purchase,
           r.helpful_vote,
           m.title AS product_title, m.price,
           m.average_rating, m.main_category, m.store
        FROM read_parquet('data/raw/reviews_raw.parquet') r
        RIGHT JOIN read_parquet('data/raw/meta_raw.parquet') m USING (parent_asin)
    )
    TO 'data/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [13]:
c2.execute(f"SELECT * FROM read_parquet('data/merged.parquet')").df()


,rating,title,text,parent_asin,verified_purchase,helpful_vote,product_title,price,average_rating,main_category,store
0,5.0,"Beautiful, heavily textured vinyl",I really like this assortment of metallic viny...,B0BW3X9XT4,False,0,Holographic Vinyl Black Rainbow Permanent Viny...,13.90,4.6,Amazon Home,GIRAFVINYL
1,2.0,Too small and seem to be better for rings,I've had a hard time using these for bracelets...,B0058EDF8C,True,0,"Eurotool Pliers, Silver",21.99,4.1,"Arts, Crafts & Sewing",EURO TOOL
2,5.0,Awesome product.,I absolutely loved being able to buy feathers ...,B07NP9VFZ3,True,1,wanjin Duck Goose Feathers Trim Fringe Craft F...,14.99,4.6,"Arts, Crafts & Sewing",Wanjin
3,5.0,Very true to color in picture. Fast shipping. ...,Beautiful beads very true to picture. Wish I h...,B00J9F6DEO,True,5,BEADNOVA 8mm White Clear Crystal Quartz Gemsto...,11.39,4.7,"Arts, Crafts & Sewing",BEADNOVA
4,5.0,Five Stars,I love this small size sketch pad. Perfect si...,B088VVG9HN,True,0,"Strathmore 300 Series Sketch Paper Pad, Glue B...",24.61,4.7,"Arts, Crafts & Sewing",Strathmore
...,...,...,...,...,...,...,...,...,...,...,...
20437,NaN,NaN,NaN,NaN,<NA>,<NA>,"Mann Lake ""Spiral Taper"" Candle Mold, 7-1/2-Inch",NaN,4.6,"Arts, Crafts & Sewing",Mann Lake
20438,NaN,NaN,NaN,NaN,<NA>,<NA>,BWRMHME 8 X 5 Inch Cross Stitch Wooden Embroid...,10.88,4.4,Amazon Home,BWRMHME
20439,NaN,NaN,NaN,NaN,<NA>,<NA>,STA Fineliner Fiber-tip Comic Sketch Rollerbal...,NaN,4.4,"Arts, Crafts & Sewing",STA
20440,NaN,NaN,NaN,NaN,<NA>,<NA>,18 Blown Speckled Jumbo Coturnix Quail Egg Shells,NaN,4.5,"Arts, Crafts & Sewing",Rose Family Farms


### Create Stratified Sample

In [14]:
OUTPUT = 'data/processed/sample.parquet'

# Use an f-string to inject the variable, and wrap it in single quotes
c2.execute(f"""
    COPY (
        SELECT DISTINCT ON (product_title) * FROM read_parquet('data/merged.parquet')
    ) TO '{OUTPUT}' (FORMAT 'PARQUET')
""")

### Convert Parquet to LangChain Documents

In [15]:
# 1. Load your newly minted stratified sample
df_sample = pd.read_parquet(OUTPUT)

# 2. Fill any random missing text values to prevent concatenation crashes
text_columns = ['product_title', 'main_category', 'text']
for col in text_columns:
    if col in df_sample.columns:
        df_sample[col] = df_sample[col].fillna("")

# 3. Engineer the hybrid page_content string
df_sample['page_content'] = (
    "Product Title: " + df_sample['product_title'] + "\n" +
    "Category: " + df_sample['main_category'] + "\n" +
    "Review Text: " + df_sample['text']
)

# 4. Drop the redundant text columns so they don't clutter the metadata
df_clean = df_sample.drop(columns=['product_title', 'main_category', 'text'])

# 5. Convert the DataFrame into LangChain Documents
documents = []
for _, row in df_clean.iterrows():
    # The engineered string becomes the searchable content
    content = row['page_content']
    
    # Every other column (rating, price, helpful_vote, rating_bucket, len_tier) 
    # gets scooped up into the metadata dictionary!
    metadata = row.drop('page_content').to_dict()
    
    # Create and append the Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Successfully created {len(documents)} LangChain Documents!")
print("\n--- Let's peek at the first one ---")
print(f"CONTENT:\n{documents[0].page_content}")
print(f"METADATA:\n{documents[0].metadata}")

Successfully created 19988 LangChain Documents!

--- Let's peek at the first one ---
CONTENT:
Product Title: Holographic Vinyl Black Rainbow Permanent Vinyl for cricut,Cameo,Adhesive Vinyl for DIY Tumblers
Category: Amazon Home
Review Text: I really like this assortment of metallic vinyl. It has a leather type texture and it really looks sharp. The attached image of the wooden ornament is fairly small, about 2x3”. It cut super easily on my Cricut. Can’t speak to how it sticks because I resined everything that I put it on. It’s definitely a rich and gorgeous look.
METADATA:
{'rating': 5.0, 'title': 'Beautiful, heavily textured vinyl', 'parent_asin': 'B0BW3X9XT4', 'verified_purchase': False, 'helpful_vote': 0.0, 'price': 13.9, 'average_rating': 4.6, 'store': 'GIRAFVINYL'}


## 1. Data Overview:

In [16]:
print(df_sample.info())
print(df_sample.columns.tolist())

<class 'pandas.DataFrame'>
RangeIndex: 19988 entries, 0 to 19987
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             897 non-null    float64
 1   title              897 non-null    str    
 2   text               19988 non-null  str    
 3   parent_asin        897 non-null    str    
 4   verified_purchase  897 non-null    object 
 5   helpful_vote       897 non-null    float64
 6   product_title      19988 non-null  str    
 7   price              11989 non-null  float64
 8   average_rating     19988 non-null  float64
 9   main_category      19988 non-null  str    
 10  store              19567 non-null  str    
 11  page_content       19988 non-null  str    
dtypes: float64(4), object(1), str(7)
memory usage: 8.2+ MB
None
['rating', 'title', 'text', 'parent_asin', 'verified_purchase', 'helpful_vote', 'product_title', 'price', 'average_rating', 'main_category', 'store', 'page_content']

#### Rating Distribution

In [17]:
# EDA example duckdb Rating distribution
c2.execute("""
    SELECT
        rating,
        COUNT(*) AS count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percent
    FROM read_parquet('data/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,count,percent
0,1.0,54,0.26
1,2.0,41,0.20
2,3.0,71,0.35
3,4.0,156,0.76
4,5.0,1017,4.98
5,NaN,19103,93.45


## 2. Model Building
- Builds **BM25 Search** model and saves it as a pickle file locally into the model's directory.
- Builds **Semantic Search** model and saves the index locally into the model's directory. 

### BM25 Search

In [18]:
# Pull documents one by one, tokenize them, and build the math model
retriever = BM25Retriever.from_documents(
    documents,
    preprocess_func=simple_tokenize
)

retriever.k = 3

# Save the BM25 index locally 
with open(bm25_path, 'wb') as f:
    pickle.dump(retriever, f)
    
print(f"Index built and safely saved to {bm25_path}!")

Index built and safely saved to models/bm25_metadata_index_big.pkl!


#### BM25 Retrieval Score
- Testing the saved BM25 model to make sure it works!

In [19]:
query = "green yarn"
print(f"\nTesting Loaded BM25 Search with query: '{query}'")

# Open the pickle file in 
with open(bm25_path, 'rb') as f:
    loaded_bm25_retriever = pickle.load(f)

# Run search using loaded model
bm25_scored_results = search_bm25_with_scores(loaded_bm25_retriever, query, k=3)
display_results(bm25_scored_results, title="Loaded BM25 Results", score_label="BM25 Score")


Testing Loaded BM25 Search with query: 'green yarn'
Loaded BM25 Results
#1 | BM25 Score: 10.166
Product: Lion Brand Yarn 860-170E Vanna's Choice Yarn, Pea Green
Rating:  No rating (0/5)
Review:  
--------------------------------------------------
#2 | BM25 Score: 9.696
Product: Lion Brand Yarn 595-205 Color Waves Yarn, Green Apple
Rating:  ⭐⭐⭐⭐⭐ (5/5)
Review:  wife loved it.
--------------------------------------------------
#3 | BM25 Score: 9.425
Product: Light Green 2 lbs 67 Yards Heavy Braid Cotton Blanket Arm Knitting Chunky Yarn for Handmade DIY Throw Blanket Pet Bed ,Washable Chunky Yarn (Light Green, 2 lbs)
Rating:  No rating (0/5)
Review:  
--------------------------------------------------


### Semantic Search

In [20]:
model = SentenceTransformer("all-MiniLM-L6-v2")
content_list = [doc.page_content for doc in documents]

document_embeddings = model.encode(content_list, show_progress_bar=True)
print(f"Successfully created {len(document_embeddings)} embeddings!")

Batches: 100%|██████████| 625/625 [00:41<00:00, 15.19it/s]


Successfully created 19988 embeddings!


In [21]:
# Format the data for LangChain
text_embedding_pairs = list(zip(content_list, document_embeddings.tolist()))
metadata_list = [doc.metadata for doc in documents]

# Query embedder
hf_query_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build the Vector Store 
vector_store = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=hf_query_embedder,
    metadatas=metadata_list
)

# Save it to disk locally
vector_store.save_local(semantic_path)
print(f"FAISS Index successfully saved to {semantic_path}!")

FAISS Index successfully saved to models/faiss_index_big!


#### Semantic Retrieval Score
- Testing the saved Semantic Search model to make sure it works!

In [22]:
print(f"\nTesting Loaded Semantic Search with query: '{query}'")

loaded_vector_store = FAISS.load_local(
    folder_path=semantic_path,
    embeddings=hf_query_embedder,
    allow_dangerous_deserialization=True
)

semantic_scored_results = search_semantic_with_scores(loaded_vector_store, query, k=3)
display_results(semantic_scored_results, title="FAISS Results", score_label="L2 Distance")


Testing Loaded Semantic Search with query: 'green yarn'
FAISS Results
#1 | L2 Distance: 0.709
Product: Light Green 2 lbs 67 Yards Heavy Braid Cotton Blanket Arm Knitting Chunky Yarn for Handmade DIY Throw Blanket Pet Bed ,Washable Chunky Yarn (Light Green, 2 lbs)
Rating:  No rating (0/5)
Review:  
--------------------------------------------------
#2 | L2 Distance: 0.723
Product: Lion Brand Yarn 860-170E Vanna's Choice Yarn, Pea Green
Rating:  No rating (0/5)
Review:  
--------------------------------------------------
#3 | L2 Distance: 0.740
Product: Premier Yarns 1050-05 Puzzle Yarn-Word Search
Rating:  No rating (0/5)
Review:  
--------------------------------------------------


## Comparison with New LLM Model

In [23]:
query1 = "sewing machine"
query2 = "acrylic paint"
query3 = "Singer 5532"
query4 = "Animal stickers"
query5 = "Purple gel pen"
query6 = "cute embroidery patterns"
query7 = "yarn made of natural fibers"
query8 = "best paint for wood"
query9 = "what's the best type of yarn to make clothes with"
query10 = "what is a good type of yarn for a beginner"

Originally chose `HuggingFace model`: `Qwen/Qwen2.5-0.5B`

In [24]:
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B",
    torch_dtype = torch.float16,
    device="mps",
    trust_remote_code=True,
    max_new_tokens=128,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=generator)

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps


In [25]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query1)

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [26]:
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: nan
Title: Mini Sewing Machine for Beginners, Model 1501S with Free Arm, Double Speed, 12 Built-in Stitch Patterns & Multi Power Modes
Rating: nan/5
Review: 

Product ASIN: nan
Title: Mini Sewing Machine for Begginers Portable Electric Crafting Mending Machine with 12 Built-in Stitches Double Thread and Speed for Beginner Blue
Rating: nan/5
Review: 

Product ASIN: nan
Title: Household Multifunctional Mini Electric Sewing Machine, Portable Sewing Machines Multi-purpose 12 Built-in Stitches With Foot Pedal For Home Sewing, Beginners, Kids
Rating: nan/5
Review: 

Product ASIN: nan
Title: Mini Electric Sewing Machine Portable Household Sewing Machine Beginner Tailors Free-Arm Crafting Mending Machine for DIY Crafting (with Table and Small Sewing Kit, US Plu

### New Model:
For our new model, we chose to use the GroqChat model "llama-3.1-8b-instant". 

In [27]:
new_llm = ChatGroq(model="llama-3.1-8b-instant")

In [28]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query1)
print(new_ensemble_answer)

It seems like you're interested in purchasing a sewing machine. Here are some top-rated options for beginners:

1. Model 1501S with Free Arm, Double Speed, 12 Built-in Stitch Patterns & Multi Power Modes - This mini sewing machine is a great option for beginners, with 12 built-in stitch patterns and double speed capabilities.
2. Portable Electric Crafting Mending Machine with 12 Built-in Stitches Double Thread and Speed for Beginner Blue - This machine is perfect for beginners who want a portable and easy-to-use sewing machine with 12 built-in stitch patterns.
3. Household Multifunctional Mini Electric Sewing Machine, Portable Sewing Machines Multi-purpose 12 Built-in Stitches With Foot Pedal For Home Sewing, Beginners, Kids - This machine is great for beginners and kids, with 12 built-in stitch patterns and a foot pedal for easy use.

Please note that these reviews and product information are based on the provided context, and actual product availability and ratings may vary.


#### Query 3

In [29]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query3)
print(old_ensemble_answer)

/Users/Nicole/miniforge3/envs/575-app/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: nan
Title: SINGER 4228 Inspiration Sewing Machine
Rating: nan/5
Review: 

Product ASIN: nan
Title: Singer 3221 Simple Sewing Machine
Rating: nan/5
Review: 

Product ASIN: nan
Title: Singer 8768 Heritage Electronic Sewing Machine
Rating: nan/5
Review: 

Product ASIN: nan
Title: Singer Simple 2263 23-Stitch Sewing Machine, White (Renewed)
Rating: nan/5
Review: 

Product ASIN: nan
Title: Singer 224121 Drip Pan
Rating: nan/5
Review: 

Product ASIN: B0092RC3YI
Title: SINGER 5532 Heavy Duty Extra-High Sewing Speed Portable Sewing Machine with Metal Frame and Stainless Steel Bedplate
Rating: 4.0/5
Review: For quite some time I thought perhaps I was missing out by not being able to sew on one of the top brand machines. I grew up learning to sew on my mother's &

In [30]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query3)
print(new_ensemble_answer)

The Singer 5532 is a 32-Stitch Heavy Duty Sewing Machine with Metal Frame and Stainless Steel Bedplate. 

It is available on Amazon with a rating of 4.0/5 stars from a customer who found it to be a nice machine with plenty of stitch options. However, the customer prefers their Brother sewing machine over the SINGER 5532 due to its lower price and more options. (ASIN: B0092RC3YI)


#### Query 5

In [31]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query5)
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: nan
Title: 8 Assorted Colored 0.4mm Fine Tip Gel Coloring Pen Set Colored Ink Pen for Writing Painting Sketching Drawing Coloring and Take Notes
Rating: nan/5
Review: 

Product ASIN: nan
Title: White Gel Ink Pens - 0.8MM Fine Tip, for Artists, Drawing, Sketching, Black Paper, Pack of 6
Rating: nan/5
Review: 

Product ASIN: nan
Title: Artlicious Deluxe 60 Unique Gel Pens Set - Non Toxic and Acid Free - Ideal for Coloring Books
Rating: nan/5
Review: 

Product ASIN: nan
Title: 60 unique colors (no repetition) gel pen set, ink volume increased by 30%, used for adult coloring book, painting, doodle, scrapbook
Rating: nan/5
Review: 

Product ASIN: nan
Title: Sakura Pigment Ink Pen, Pigma Micron 08, Green (ESDK08#29)
Rating: nan/5
Review: 

Product ASIN: nan
T

In [32]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query5)
print(new_ensemble_answer)

I couldn't find a product that directly matches "Purple gel pen". However, I can suggest a product that might be related to your search.

The "Artlicious Deluxe 60 Unique Gel Pens Set" (no ASIN provided) is a set of gel pens that includes a variety of colors, but it's not specified if it includes a purple gel pen specifically. Another product, "Art School Gel Pens - 36 Gel Pen Set and Glitter Gel Pens for Adult Coloring Books" (no ASIN provided), is a set of gel pens, but it's not clear if it includes purple gel pens either.

If you're looking for a specific brand or type of purple gel pen, please let me know and I can try to help you further.

Also, I can suggest that you might be interested in the "YAOYOU 8 Assorted Colored Gel Coloring Pen Set" or the "White Gel Ink Pens - 0.8MM Fine Tip" (no ASIN provided), which are sets of gel pens with assorted colors, but not specifically purple.


### Query 7

In [33]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query7)
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B0BWSF1HPY
Title: Arm Knitting Yarn for Chunky Braided Knot Throw Blanket DIY, Soft Extra Cotton Washable Tube Bulky Giant Yarn for Weave Craft Crochet (Dark Gray 2.2lb)
Rating: 5.0/5
Review: [[VIDEOID:21dfc200bc65819023d8994064a70121]] Seems like simple and normal yarn.  The feel was pretty soft I thought.<br /><br />Color seems accurate to the pictures.  Felt like what you’d use to make sweaters or something similar.

Product ASIN: nan
Title: Premier Yarns 1050-05 Puzzle Yarn-Word Search
Rating: nan/5
Review: 

Product ASIN: B07T6C5ND4
Title: Bernat Handicrafter Cotton Yarn, Gauge 4 Medium Worsted, Salt/Pepper
Rating: 4.0/5
Review: This cotton yarn is a decent option for making dish cloths and pot holders, but I would not recommend it for anything wea

In [34]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query7)
print(new_ensemble_answer)

Based on the provided context, I found a few options that match your requirement:

1. **Premier Yarns 1050-05 Puzzle Yarn-Word Search** (no ASIN provided): This yarn is made of natural fibers, although the type is not specified. However, Premier Yarns is a well-known brand that offers a range of natural fiber yarns.

2. **Gutermann 100% Natural Cotton Thread - 110 Yds - Color 7310** (no ASIN provided): As the name suggests, this yarn is made of 100% natural cotton, a fiber that is biodegradable and gentle on the environment.

3. **Threadart 100% Pure Cotton Crochet Yarn | Dark Green | 50 Gram Skeins | Worsted Medium #4 Yarn | 85 yds per Skein** (no ASIN provided): This yarn is also made of 100% natural cotton, making it an eco-friendly option for crafters.

4. **Revolution Fibers | Undyed (Off-White) BFL Wool Yarn | Worsted (Aran) Weight Yarn Hank | 100 Grams, Approx 175 Yards, 165 Meters** (no ASIN provided): This yarn is made of BFL (Bluefaced Leicester) wool, a natural fiber known f

#### Query 9

In [35]:
old_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, llm)

old_ensemble_answer = old_rag_chain.invoke(query9)
print(old_ensemble_answer)

Human: 
    You are a helpful Amazon shopping assistant.
        Answer the question using ONLY the following context (real product reviews + metadata).
        Always cite the product ASIN when possible.

    Context:
    Product ASIN: B017QU7N7Y
Title: Lion Brand Yarn 5800-526 Martha Stewart Glitter Eyelash Yarn, Brownstone
Rating: 4.0/5
Review: This pretty, sparkly eyelash yarn  adds a lovely, glamorous  look to to various projects.  The yarn has a touch of delicate pink, and also reflects sparkly rainbow colors.  I am just learning to work with eyelash yarn, but can say it adds a lovely accent to a hat, or scarf.  It can be a bit tricky to work with at first, but you will pick it up as you go along.  For me, it works better in knit projects then as a crochet yarn.  Make sure you note the weight and yardage.  These are tiny skeins, so don't be taken by surprise by that.  It is a bulky yarn, so that is another thing to consider in picking a project.  I like it.

Product ASIN: B0BWSF1

In [36]:
new_rag_chain = get_rag_chain(loaded_bm25_retriever, loaded_vector_store, new_llm)

new_ensemble_answer = new_rag_chain.invoke(query9)
print(new_ensemble_answer)

Based on the provided reviews, it seems that the best type of yarn to make clothes with is a high-quality, mercerized cotton yarn. 

The review for product ASIN: B099XDSQCY (24/7 Cotton Yarn, Ecru) mentions that it is a wonderful cotton yarn that is high quality, easy to work with, long-lasting, and versatile. It is also mentioned that it leans toward a strand of yarn being DK (3) size yarn, which is suitable for making clothes.

Additionally, the review for product ASIN: B07T6C5ND4 (Bernat Handicrafter Cotton Yarn, Gauge 4 Medium Worsted, Salt/Pepper) mentions that the yarn is a decent option for making dish cloths and pot holders, but not recommended for anything wearable. However, it is worth noting that this yarn is a medium-worsted weight yarn, which may not be suitable for making clothes.

The best option for making clothes would be a yarn that is lightweight, breathable, and durable, such as a mercerized cotton yarn.
